<a href="https://colab.research.google.com/github/mak-shah/COO/blob/master/quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

In [ ]:
def linear_q_with_scale_and_zero_point(
    tensor, scale, zero_point, dtype=torch.int8):
    """Quantize a floating-point tensor to integer representation."""

    scaled_and_shifted_tensor = tensor / scale + zero_point

    rounded_tensor = torch.round(scaled_and_shifted_tensor)

    q_min = torch.iinfo(dtype).min
    q_max = torch.iinfo(dtype).max

    q_tensor = rounded_tensor.clamp(q_min, q_max).to(dtype)

    return q_tensor

In [ ]:
def linear_dequantization(quantized_tensor, scale, zero_point):
    """Dequantize an integer tensor back to floating point."""
    return scale * (quantized_tensor.float() - zero_point)

In [ ]:
### a dummy tensor to test the implementation
test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6],
     [92.14, 295.5,  -184],
     [0,     684.6, 245.5]]
)

### these are random values for "scale" and "zero_point"
scale = 3.5
zero_point = -70

quantized_tensor = linear_q_with_scale_and_zero_point(
    test_tensor, scale, zero_point)

print("Quantized (INT8):")
print(quantized_tensor)
print(f"\nDtype: {quantized_tensor.dtype}")

Quantized (INT8):
tensor([[ -15,  -74,  127],
        [ -44,   14, -123],
        [ -70,  126,    0]], dtype=torch.int8)

Dtype: torch.int8


In [ ]:
dequantized_tensor = linear_dequantization(
    quantized_tensor, scale, zero_point)

print("Original:")
print(test_tensor)
print("\nDequantized:")
print(dequantized_tensor)

Original:
tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])

Dequantized:
tensor([[ 192.5000,  -14.0000,  689.5000],
        [  91.0000,  294.0000, -185.5000],
        [   0.0000,  686.0000,  245.0000]])


In [ ]:
# Element-wise error
print("Difference (Original - Dequantized):")
print(dequantized_tensor - test_tensor)

print("\nSquared error:")
print((dequantized_tensor - test_tensor).square())

# Overall error via MSE
mse = (dequantized_tensor - test_tensor).square().mean()
print(f"\nQuantization Error (MSE): {mse.item():.4f}")

Difference (Original - Dequantized):
tensor([[  0.9000,  -0.5000, -39.1000],
        [ -1.1400,  -1.5000,  -1.5000],
        [  0.0000,   1.4000,  -0.5000]])

Squared error:
tensor([[8.0999e-01, 2.5000e-01, 1.5288e+03],
        [1.2996e+00, 2.2500e+00, 2.2500e+00],
        [0.0000e+00, 1.9601e+00, 2.5000e-01]])

Quantization Error (MSE): 170.8753


In [ ]:
def get_q_scale_and_zero_point(tensor, dtype=torch.int8):
    """Compute optimal scale and zero point for asymmetric quantization."""

    q_min, q_max = torch.iinfo(dtype).min, torch.iinfo(dtype).max
    r_min, r_max = tensor.min().item(), tensor.max().item()

    scale = (r_max - r_min) / (q_max - q_min)

    zero_point = q_min - (r_min / scale)

    # clip the zero_point to fall in [quantized_min, quantized_max]
    if zero_point < q_min:
        zero_point = q_min
    elif zero_point > q_max:
        zero_point = q_max
    else:
        # round and cast to int
        zero_point = int(round(zero_point))

    return scale, zero_point

In [ ]:
scale, zero_point = get_q_scale_and_zero_point(test_tensor)
print(f"Computed scale:      {scale:.6f}")
print(f"Computed zero_point: {zero_point}")

Computed scale:      3.578823
Computed zero_point: -77


In [ ]:
# Quantize with computed parameters
quantized_tensor = linear_q_with_scale_and_zero_point(
    test_tensor, scale, zero_point)

dequantized_tensor = linear_dequantization(
    quantized_tensor, scale, zero_point)

mse = (dequantized_tensor - test_tensor).square().mean()

print(f"MSE with computed params: {mse.item():.4f}")
print(f"\nOriginal:\n{test_tensor}")
print(f"\nDequantized:\n{dequantized_tensor}")


MSE with computed params: 1.5730

Original:
tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])

Dequantized:
tensor([[ 193.2565,  -14.3153,  730.0800],
        [  93.0494,  297.0423, -182.5200],
        [   0.0000,  683.5552,  246.9388]])


In [ ]:
def linear_quantization(tensor, dtype=torch.int8):
    """End-to-end linear quantization: compute params and quantize."""
    scale, zero_point = get_q_scale_and_zero_point(tensor, dtype=dtype)

    quantized_tensor = linear_q_with_scale_and_zero_point(
        tensor, scale, zero_point, dtype=dtype)

    return quantized_tensor, scale, zero_point

In [ ]:
# Test on a random tensor
r_tensor = torch.randn((4, 4))
print("Original tensor:")
print(r_tensor)

quantized_tensor, scale, zero_point = linear_quantization(r_tensor)
print(f"\nQuantized tensor (INT8):")
print(quantized_tensor)
print(f"\nScale: {scale:.6f}, Zero Point: {zero_point}")

dequantized_tensor = linear_dequantization(quantized_tensor, scale, zero_point)
mse = (dequantized_tensor - r_tensor).square().mean()
print(f"\nQuantization Error (MSE): {mse.item():.6f}")

Original tensor:
tensor([[ 1.0324,  1.3197,  1.3159,  1.2389],
        [-0.4339,  0.4959,  0.4776,  1.0897],
        [-0.3041, -0.4898,  2.1773, -0.3143],
        [-1.7095,  0.6097,  0.7488, -0.2499]])

Quantized tensor (INT8):
tensor([[  52,   71,   70,   65],
        [ -44,   17,   15,   55],
        [ -36,  -48,  127,  -37],
        [-128,   24,   33,  -32]], dtype=torch.int8)

Scale: 0.015242, Zero Point: -16

Quantization Error (MSE): 0.000023


In [ ]:
# Symmetric Quantization (absmax)
def get_q_scale_symmetric(tensor, dtype=torch.int8):
    """Compute scale for symmetric (absmax) quantization."""
    q_max = torch.iinfo(dtype).max  # 127 for int8
    alpha = tensor.abs().max().item()
    scale = alpha / q_max
    return scale


def linear_q_symmetric(tensor, dtype=torch.int8):
    """Symmetric quantization using absmax."""
    scale = get_q_scale_symmetric(tensor, dtype)
    quantized_tensor = linear_q_with_scale_and_zero_point(
        tensor, scale, zero_point=0, dtype=dtype)
    return quantized_tensor, scale

In [ ]:
# Symmetric quantization on test_tensor
quantized_tensor, scale = linear_q_symmetric(test_tensor)
dequantized_tensor = linear_dequantization(quantized_tensor, scale, 0)
mse = (dequantized_tensor - test_tensor).square().mean()

print(f"Scale (symmetric): {scale:.6f}")
print(f"Quantization Error (MSE): {mse.item():.4f}")
print(f"\nOriginal:\n{test_tensor}")
print(f"\nDequantized:\n{dequantized_tensor}")

Scale (symmetric): 5.737008
Quantization Error (MSE): 2.5092

Original:
tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])

Dequantized:
tensor([[ 189.3213,  -11.4740,  728.6000],
        [  91.7921,  298.3244, -183.5842],
        [   0.0000,  682.7039,  246.6913]])


In [ ]:
# A small calibration dataset (representative inputs)
calibration_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a Python function to sort a list.",
    "What are the benefits of exercise?",
    "Summarize the plot of Romeo and Juliet.",
    "How does photosynthesis work?",
]
print(f"Calibration samples: {len(calibration_prompts)}")


Calibration samples: 5


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32, device_map=DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on cuda


In [ ]:
# Storage for activation statistics
activation_stats = {}

def make_hook(name):
    """Create a hook that records min/max of layer outputs."""
    def hook_fn(module, input, output):
        # Handle tuple outputs (common in transformer layers)
        if isinstance(output, tuple):
            tensor = output[0]
        else:
            tensor = output

        if name not in activation_stats:
            activation_stats[name] = {"min": float("inf"), "max": float("-inf")}

        activation_stats[name]["min"] = min(
            activation_stats[name]["min"], tensor.min().item())
        activation_stats[name]["max"] = max(
            activation_stats[name]["max"], tensor.max().item())
    return hook_fn

# Register hooks on the first 4 transformer layers
hooks = []
for i in range(4):
    layer = model.model.layers[i]
    h = layer.register_forward_hook(make_hook(f"layer_{i}"))
    hooks.append(h)

print(f"Hooks registered on {len(hooks)} layers")

Hooks registered on 4 layers


In [ ]:
# Run calibration inference (no gradients needed)
with torch.no_grad():
    for prompt in calibration_prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        _ = model(**inputs)

print("Calibration inference complete.")
print(f"Collected stats for {len(activation_stats)} layers")


Calibration inference complete.
Collected stats for 4 layers


In [ ]:
# Compute scale and zero-point from collected statistics
print(f"{'Layer':<12} {'Min':>10} {'Max':>10} {'Scale':>12} {'Zero Point':>12}")
print("=" * 58)

for name, stats in activation_stats.items():
    r_min = stats["min"]
    r_max = stats["max"]

    # Asymmetric quantization parameters for INT8
    q_min, q_max = -128, 127
    scale = (r_max - r_min) / (q_max - q_min)
    zero_point = int(round(q_min - r_min / scale))
    zero_point = max(q_min, min(q_max, zero_point))  # clip

    print(f"{name:<12} {r_min:>10.3f} {r_max:>10.3f} {scale:>12.6f} {zero_point:>12d}")

# Clean up hooks
for h in hooks:
    h.remove()

# Free the model
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Layer               Min        Max        Scale   Zero Point
layer_0          -0.460      0.498     0.003756           -6
layer_1          -0.668      0.840     0.005915          -15
layer_2        -140.672     21.588     0.636314           93
layer_3        -140.717     21.602     0.636544           93


In [ ]:
def linear_q_symmetric_per_channel(tensor, dim=0, dtype=torch.int8):
    """Symmetric quantization with a separate scale per channel."""

    output_dim = tensor.shape[dim]
    scale = torch.zeros(output_dim)

    for i in range(output_dim):
        sub_tensor = tensor.select(dim, i)
        scale[i] = get_q_scale_symmetric(sub_tensor, dtype)

    # Reshape scale for broadcasting
    shape = [1] * tensor.dim()
    shape[dim] = -1
    scale = scale.view(shape)

    quantized_tensor = linear_q_with_scale_and_zero_point(
        tensor, scale, zero_point=0, dtype=dtype)

    return quantized_tensor, scale

In [ ]:
def quantization_error(original, dequantized):
    return (original - dequantized).square().mean()

# Compare per-tensor vs per-channel
qt_t, s_t = linear_q_symmetric(test_tensor)
dq_t = linear_dequantization(qt_t, s_t, 0)
mse_t = quantization_error(test_tensor, dq_t).item()

qt_c, s_c = linear_q_symmetric_per_channel(test_tensor, dim=0)
dq_c = linear_dequantization(qt_c, s_c, 0)
mse_c = quantization_error(test_tensor, dq_c).item()

print(f"Per-Tensor  MSE: {mse_t:.4f}")
print(f"Per-Channel MSE: {mse_c:.4f}")
print(f"\nPer-channel reduces error by {(1 - mse_c/mse_t)*100:.1f}%")


Per-Tensor  MSE: 2.5092
Per-Channel MSE: 1.8084

Per-channel reduces error by 27.9%


In [ ]:
def linear_q_symmetric_per_group(tensor, group_size, dtype=torch.int8):
    """Symmetric quantization with a separate scale per group."""

    t_shape = tensor.shape
    assert t_shape[1] % group_size == 0
    assert tensor.dim() == 2

    tensor = tensor.view(-1, group_size)

    quantized_tensor, scale = linear_q_symmetric_per_channel(
        tensor, dim=0, dtype=dtype)

    quantized_tensor = quantized_tensor.view(t_shape)

    return quantized_tensor, scale


def linear_dequantization_per_group(quantized_tensor, scale, group_size):
    """Dequantize a per-group quantized tensor."""

    q_shape = quantized_tensor.shape
    quantized_tensor = quantized_tensor.view(-1, group_size)

    dequantized_tensor = linear_dequantization(quantized_tensor, scale, 0)

    dequantized_tensor = dequantized_tensor.view(q_shape)

    return dequantized_tensor

In [ ]:
# Test per-group quantization
test_tensor_6x6 = torch.randn((6, 6))
group_size = 3

qt_g, s_g = linear_q_symmetric_per_group(
    test_tensor_6x6, group_size=group_size)

dq_g = linear_dequantization_per_group(qt_g, s_g, group_size)
mse_g = quantization_error(test_tensor_6x6, dq_g).item()

# Compare with per-tensor
qt_t2, s_t2 = linear_q_symmetric(test_tensor_6x6)
dq_t2 = linear_dequantization(qt_t2, s_t2, 0)
mse_t2 = quantization_error(test_tensor_6x6, dq_t2).item()

print(f"Per-Tensor MSE:              {mse_t2:.6f}")
print(f"Per-Group  MSE (group={group_size}):  {mse_g:.6f}")
print(f"\nScales stored for per-tensor: 1")
print(f"Scales stored for per-group:  {s_g.numel()}")

Per-Tensor MSE:              0.000025
Per-Group  MSE (group=3):  0.000007

Scales stored for per-tensor: 1
Scales stored for per-group:  12


In [ ]:
!pip install -q optimum-quanto transformers accelerate torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.5 MB/s eta 0:00:00


In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import quantize, freeze, qint8

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
SEED     = 42

print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
def get_model_size_mb(model):
    """Get model size in MB by summing parameter memory."""
    total = sum(p.nelement() * p.element_size() for p in model.parameters())
    return total / (1024 * 1024)


def generate_text(model, tokenizer, prompt, max_new_tokens=50):
    """Generate text and return output + latency."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False)
    latency = time.time() - start

    generated = tokenizer.decode(output[0][inputs.input_ids.shape[1]:],
                                 skip_special_tokens=True)
    return generated, latency

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp32 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32, device_map=DEVICE)

fp32_size = get_model_size_mb(model_fp32)
print(f"FP32 model size: {fp32_size:.1f} MB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FP32 model size: 4196.4 MB


In [ ]:
prompt = "Explain quantization in machine learning in 2 sentences."

fp32_output, fp32_latency = generate_text(model_fp32, tokenizer, prompt)

print(f"FP32 latency: {fp32_latency:.2f}s")
print(f"FP32 output:\n{fp32_output}")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FP32 latency: 1.62s
FP32 output:
Quantization is a technique used in machine learning to reduce the size of the input data without losing any information. It involves converting the input data into a smaller range of values, typically between 0 and 1, to reduce the computational complexity of the
